## Business Questions via SQL — Telco Customer Churn

## Why this notebook exists

Before any feature engineering or model training happens, we will answer five core
business questions directly from the database using SQL, before committing time to a
model. This validates whether real signal exists in the data.

## The data

The raw dataset (IBM Telco Customer Churn, 7,043 customers × 21 columns) has
been split across 4 normalized Postgres tables, linked by `customer_id`:

**`customers`** — who the customer is
| Column | Type | Description |
|---|---|---|
| customer_id | VARCHAR | Unique customer identifier (primary key) |
| gender | VARCHAR | Male / Female |
| senior_citizen | BOOLEAN | Whether the customer is a senior citizen |
| partner | BOOLEAN | Has a partner |
| dependents | BOOLEAN | Has dependents |
| tenure | INTEGER | Months the customer has stayed with the company |

**`services`** — what the customer has subscribed to
| Column | Type | Description |
|---|---|---|
| customer_id | VARCHAR | Foreign key → customers |
| phone_service | BOOLEAN | Has phone service |
| multiple_lines | VARCHAR | Yes / No / No phone service |
| internet_service | VARCHAR | DSL / Fiber optic / No |
| online_security | VARCHAR | Yes / No / No internet service |
| online_backup | VARCHAR | Yes / No / No internet service |
| device_protection | VARCHAR | Yes / No / No internet service |
| tech_support | VARCHAR | Yes / No / No internet service |
| streaming_tv | VARCHAR | Yes / No / No internet service |
| streaming_movies | VARCHAR | Yes / No / No internet service |

**`billing`** — contract and payment details
| Column | Type | Description |
|---|---|---|
| customer_id | VARCHAR | Foreign key → customers ||
| contract | VARCHAR | Month-to-month / One year / Two year |
| paperless_billing | BOOLEAN | Uses paperless billing |
| payment_method | VARCHAR | Electronic check / Mailed check / Bank transfer / Credit card |
| monthly_charges | NUMERIC | Current monthly charge amount |
| total_charges | NUMERIC | Total amount charged over the customer's tenure |

**`churn_status`** — the target
| Column | Type | Description |
|---|---|---|
| customer_id | VARCHAR | Foreign key → customers |
| churn | BOOLEAN | Whether the customer churned |

## The five business questions

1. **Do month-to-month contract customers churn more than annual contract customers?**
2. **Does higher monthly charge correlate with higher churn risk?**
3. **How much does tenure reduce churn risk?**
4. **Which service features are most associated with retention?**
5. **If we retain 50% of predicted churners with a discount, how much revenue is saved?**

Each question is answered with a SQL query run against Postgres, pulled into
a pandas DataFrame via `pd.read_sql()`, closing
with a short written insight before moving to the next question.

In [7]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv


In [53]:
## connecting to database 

load_dotenv(dotenv_path="../.env")

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)


In [54]:
# lets see how tables look like 
# Fetch first 5 rows of customers  table
df_customers = pd.read_sql("SELECT * FROM customers LIMIT 5;", engine)
df_customers

,customer_id,gender,senior_citizen,partner,dependents,tenure
0,7590-VHVEG,Female,False,True,False,1
1,5575-GNVDE,Male,False,False,False,34
2,3668-QPYBK,Male,False,False,False,2
3,7795-CFOCW,Male,False,False,False,45
4,9237-HQITU,Female,False,False,False,2


In [55]:
# Fetch first 5 rows of services table
df_services = pd.read_sql("SELECT * FROM services LIMIT 5;", engine)
df_services

,customer_id,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies
0,7590-VHVEG,False,No phone service,DSL,No,Yes,No,No,No,No
1,5575-GNVDE,True,No,DSL,Yes,No,Yes,No,No,No
2,3668-QPYBK,True,No,DSL,Yes,Yes,No,No,No,No
3,7795-CFOCW,False,No phone service,DSL,Yes,No,Yes,Yes,No,No
4,9237-HQITU,True,No,Fiber optic,No,No,No,No,No,No


In [56]:
df_billing = pd.read_sql("SELECT * FROM billing LIMIT 5;", engine)
df_billing

,customer_id,contract,paperless_billing,payment_method,monthly_charges,total_charges
0,7590-VHVEG,Month-to-month,True,Electronic check,29.85,29.85
1,5575-GNVDE,One year,False,Mailed check,56.95,1889.50
2,3668-QPYBK,Month-to-month,True,Mailed check,53.85,108.15
3,7795-CFOCW,One year,False,Bank transfer (automatic),42.30,1840.75
4,9237-HQITU,Month-to-month,True,Electronic check,70.70,151.65


In [57]:
df_churn = pd.read_sql("SELECT * FROM churn_status LIMIT 5;", engine)
df_churn

,customer_id,churn
0,7590-VHVEG,False
1,5575-GNVDE,False
2,3668-QPYBK,True
3,7795-CFOCW,False
4,9237-HQITU,True


## 1.Do month-to-month contract customers churn more than annual contract customers?

In [58]:


query_1 = """ SELECT b.contract , 
                     COUNT(c.customer_id) AS total_customers , 
                     SUM( CASE WHEN cs.churn THEN 1 ELSE 0 END) AS churned_customers ,
                     ROUND(SUM(CASE WHEN cs.churn THEN 1 ELSE 0 END) *100.0 / COUNT(c.customer_id) , 2 ) AS churn_rate
              FROM billing as b 
              JOIN customers c ON b.customer_id = c.customer_id
              JOIN churn_status cs on b.customer_id = cs.customer_id
              GROUP BY b.contract;
         """

df_1 = pd.read_sql(query_1 , engine)
df_1

,contract,total_customers,churned_customers,churn_rate
0,One year,1473,166,11.27
1,Month-to-month,3875,1655,42.71
2,Two year,1695,48,2.83


**Observation--** 
Month-to-month customers churn at 42.7%, versus 11.3% for one-year and 2.8% for two-year contracts. That's roughly 15x the churn rate of two-year customers, so contract type is one of the strongest churn signals in the data. Month-to-month customers also make up 55% of the base, so the exposure is large.

## 2. Does higher monthly charge correlate with higher churn risk?

In [59]:
query_2 = """
SELECT 
   CASE
    WHEN b.monthly_charges < 20 THEN 'Low (< 20)'
    WHEN b.monthly_charges < 80 THEN 'Medium (20-80)'
    ELSE 'High (80+)'
END AS charge_tier,
    COUNT(b.customer_id) AS total_customers,
    SUM(CASE WHEN cs.churn THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(SUM(CASE WHEN cs.churn THEN 1 ELSE 0 END) * 100.0 / COUNT(b.customer_id), 2) AS churn_rate_pct
FROM billing b
JOIN churn_status cs ON b.customer_id = cs.customer_id
GROUP BY charge_tier
ORDER BY total_customers DESC;
"""

df_2 = pd.read_sql(query_2, engine)
df_2

,charge_tier,total_customers,churned_customers,churn_rate_pct
0,Medium (20-80),3753,904,24.09
1,High (80+),2677,910,33.99
2,Low (< 20),613,55,8.97


**Observation --** Churn rises with price: 9.0% in the low tier, 24.1% in the medium tier, 34.0% in the high tier. Higher-paying customers are clearly at more risk. Caveat: charges are correlated with services (fiber internet, streaming add-ons), so price may be a proxy for those rather than the cause.

## 3.How much does tenure reduce churn risk?

In [60]:
query_3 = """
SELECT 
    CASE 
        WHEN c.tenure <= 12 THEN '0-1 Year'
        WHEN c.tenure BETWEEN 13 AND 24 THEN '1-2 Years'
        WHEN c.tenure BETWEEN 25 AND 48 THEN '2-4 Years'
        WHEN c.tenure BETWEEN 49 AND 60 THEN '4-5 Years'
        ELSE '5+ Years'
    END AS tenure_tier,
    COUNT(c.customer_id) AS total_customers,
    SUM(CASE WHEN cs.churn THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(SUM(CASE WHEN cs.churn THEN 1 ELSE 0 END) * 100.0 / COUNT(c.customer_id), 2) AS churn_rate_pct
FROM customers c
JOIN churn_status cs ON c.customer_id = cs.customer_id
GROUP BY tenure_tier
ORDER BY tenure_tier;
"""

df_3 = pd.read_sql(query_3, engine)
df_3

,tenure_tier,total_customers,churned_customers,churn_rate_pct
0,0-1 Year,2186,1037,47.44
1,1-2 Years,1024,294,28.71
2,2-4 Years,1594,325,20.39
3,4-5 Years,832,120,14.42
4,5+ Years,1407,93,6.61


**Observation --** Churn falls steadily with tenure, from 47.4% in the first year to 6.6% after five years. The first 12 months are the danger zone, where 1,037 of the 1,869 total churners (55%) leave. Retention efforts should target new customers.

## 4.Which service features are most associated with retention?

In [61]:
query_4 = """
SELECT 
    'Online Security' AS service_feature,
    s.online_security AS status,
    COUNT(s.customer_id) AS total_customers,
    SUM(CASE WHEN cs.churn THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(SUM(CASE WHEN cs.churn THEN 1 ELSE 0 END) * 100.0 / COUNT(s.customer_id), 2) AS churn_rate_pct
FROM services s
JOIN churn_status cs ON s.customer_id = cs.customer_id
GROUP BY s.online_security

UNION ALL

SELECT 
    'Tech Support' AS service_feature,
    s.tech_support AS status,
    COUNT(s.customer_id) AS total_customers,
    SUM(CASE WHEN cs.churn THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(SUM(CASE WHEN cs.churn THEN 1 ELSE 0 END) * 100.0 / COUNT(s.customer_id), 2) AS churn_rate_pct
FROM services s
JOIN churn_status cs ON s.customer_id = cs.customer_id
GROUP BY s.tech_support

UNION ALL

SELECT 
    'Online Backup' AS service_feature,
    s.online_backup AS status,
    COUNT(s.customer_id) AS total_customers,
    SUM(CASE WHEN cs.churn THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(SUM(CASE WHEN cs.churn THEN 1 ELSE 0 END) * 100.0 / COUNT(s.customer_id), 2) AS churn_rate_pct
FROM services s
JOIN churn_status cs ON s.customer_id = cs.customer_id
GROUP BY s.online_backup

UNION ALL

SELECT 
    'Device Protection' AS service_feature,
    s.device_protection AS status,
    COUNT(s.customer_id) AS total_customers,
    SUM(CASE WHEN cs.churn THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(SUM(CASE WHEN cs.churn THEN 1 ELSE 0 END) * 100.0 / COUNT(s.customer_id), 2) AS churn_rate_pct
FROM services s
JOIN churn_status cs ON s.customer_id = cs.customer_id
GROUP BY s.device_protection

ORDER BY service_feature, status;
"""

df_4 = pd.read_sql(query_4, engine)
df_4

,service_feature,status,total_customers,churned_customers,churn_rate_pct
0,Device Protection,No,3095,1211,39.13
1,Device Protection,No internet service,1526,113,7.40
2,Device Protection,Yes,2422,545,22.50
3,Online Backup,No,3088,1233,39.93
4,Online Backup,No internet service,1526,113,7.40
5,Online Backup,Yes,2429,523,21.53
6,Online Security,No,3498,1461,41.77
7,Online Security,No internet service,1526,113,7.40
8,Online Security,Yes,2019,295,14.61
9,Tech Support,No,3473,1446,41.64


**Observation --** Customers with Online Security or Tech Support churn at about 15%, versus about 42% without, the largest gaps of any feature (27 points). Online Backup and Device Protection show smaller gaps (18 points). Customers with no internet service churn least (7.4%), but that's a different population, so it isn't comparable.

## 5.If we retain 50% of predicted churners with a discount, how much revenue is saved?

In [62]:
query_5 = """
WITH assumptions AS (
    SELECT
        0.50 AS retention_rate,      -- share of churners we successfully retain
        0.20 AS discount_rate,       -- discount given to retained customers
        12   AS months_retained      -- how long retained customers stay
),
churner_revenue AS (
    SELECT
        COUNT(*)                        AS total_churners,
        SUM(b.monthly_charges)          AS lost_monthly_revenue
    FROM billing b
    JOIN churn_status cs ON b.customer_id = cs.customer_id
    WHERE cs.churn = TRUE
)
SELECT
    cr.total_churners,
    ROUND(cr.lost_monthly_revenue::numeric, 2)                                    AS lost_monthly_revenue,
    ROUND((cr.total_churners * a.retention_rate)::numeric, 0)                     AS customers_retained,
    ROUND((cr.lost_monthly_revenue * a.retention_rate * a.months_retained)::numeric, 2)
                                                                                  AS revenue_kept_before_discount,
    ROUND((cr.lost_monthly_revenue * a.retention_rate * a.months_retained
           * a.discount_rate)::numeric, 2)                                        AS discount_cost,
    ROUND((cr.lost_monthly_revenue * a.retention_rate * a.months_retained
           * (1 - a.discount_rate))::numeric, 2)                                  AS net_revenue_saved
FROM churner_revenue cr
CROSS JOIN assumptions a;
"""

df_5 = pd.read_sql(query_5, engine)
df_5

,total_churners,lost_monthly_revenue,customers_retained,revenue_kept_before_discount,discount_cost,net_revenue_saved
0,1869,139130.85,935.0,834785.1,166957.02,667828.08


**Observation --** Churners represent about `$139,131` in lost monthly revenue. Assuming we retain 50% of them for 12 months and give a 20% discount, the net revenue saved comes to roughly `$667,828`, after about `$166,957` in discount cost.

**Assumptions and limits:** This estimate assumes every retained customer stays a full 12 months at their current price, and that every churner is equally easy to retain. In reality, retention offers work better on some customers than others, so treating all churners the same is a simplification. This is the gap my model will address: instead of retaining 50% of churners at random, I'll target the customers with the highest predicted churn probability, and I'll quantify that improvement on Day when I calculate the business impact of the model.